In [1]:
# SMS Spam Classifier - Cleaned Code

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import nltk
import string
from nltk.corpus import stopwords
from nltk.stem.porter import PorterStemmer
from nltk.tokenize import word_tokenize, sent_tokenize
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, confusion_matrix, precision_score, classification_report
import pickle

# 1. Load Dataset
df = pd.read_csv('spam.csv', encoding='ISO-8859-1')
df.drop(columns=['Unnamed: 2','Unnamed: 3','Unnamed: 4'], inplace=True)
df.rename(columns={'v1':'target', 'v2':'text'}, inplace=True)

# 2. Encode Labels
encoder = LabelEncoder()
df['target'] = encoder.fit_transform(df['target'])

# 3. Remove Duplicates
df = df.drop_duplicates(keep='first')

# 4. Feature Engineering
# 4. Feature Engineering
nltk.download("punkt")
nltk.download("punkt_tab")

df['num_characters'] = df['text'].apply(len)
df['num_words'] = df['text'].apply(lambda x: len(word_tokenize(x)))
df['num_sentences'] = df['text'].apply(lambda x: len(sent_tokenize(x)))


# 5. Preprocessing Function
nltk.download('stopwords')
ps = PorterStemmer()

def transform_text(text):
    text = text.lower()
    text = word_tokenize(text)
    y = [i for i in text if i.isalnum()]
    y = [i for i in y if i not in stopwords.words('english') and i not in string.punctuation]
    y = [ps.stem(i) for i in y]
    return " ".join(y)

df['transformed_text'] = df['text'].apply(transform_text)

# 6. TF-IDF Vectorization
tfidf = TfidfVectorizer(max_features=3000)
x = tfidf.fit_transform(df['transformed_text']).toarray()
y = df['target'].values

# 7. Train-Test Split
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=2)

# 8. Model Training (Multinomial Naive Bayes)
mnb = MultinomialNB()
mnb.fit(x_train, y_train)
y_pred = mnb.predict(x_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("Classification Report:\n", classification_report(y_test, y_pred))

# 9. Save Model and Vectorizer
pickle.dump(tfidf, open('vectorizer.pkl', 'wb'))
pickle.dump(mnb, open('model.pkl', 'wb'))


[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\DELL\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\DELL\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping tokenizers\punkt_tab.zip.
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\DELL\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


Accuracy: 0.9709864603481625
Precision: 1.0
Confusion Matrix:
 [[896   0]
 [ 30 108]]
Classification Report:
               precision    recall  f1-score   support

           0       0.97      1.00      0.98       896
           1       1.00      0.78      0.88       138

    accuracy                           0.97      1034
   macro avg       0.98      0.89      0.93      1034
weighted avg       0.97      0.97      0.97      1034



In [2]:
# Load vectorizer & model if needed
# tfidf = pickle.load(open('vectorizer.pkl', 'rb'))
# mnb = pickle.load(open('model.pkl', 'rb'))

def check_message(message):
    # 1. Transform text (same preprocessing as before)
    transformed = transform_text(message)
    
    # 2. Convert to vector
    vector_input = tfidf.transform([transformed])
    
    # 3. Predict
    result = mnb.predict(vector_input)[0]
    
    return "Spam 🚨" if result == 1 else "Not Spam ✅"

# Test messages
print(check_message("Congratulations! You've won a free lottery ticket. Claim now!"))
print(check_message("Hey bro, are we meeting for cricket practice tomorrow?"))
print(check_message("Exclusive deal only today! Click here to win a prize."))
print(check_message("Please submit the assignment before midnight."))


Spam 🚨
Not Spam ✅
Spam 🚨
Not Spam ✅
